In [7]:
import os, time, json, math
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Any, Optional
from scipy import stats
from pathlib import Path
import re
from difflib import SequenceMatcher
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.fuzz import WRatio
from rapidfuzz.distance import Levenshtein
import textdistance
import datetime 


from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_community.callbacks import get_openai_callback

from PyDI.entitymatching.blocking import TokenBlocker, StandardBlocker, SortedNeighbourhoodBlocker, EmbeddingBlocker
from PyDI.entitymatching.comparators import StringComparator, NumericComparator
from PyDI.entitymatching.feature_extraction import FeatureExtractor
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator
from PyDI.schemamatching import LLMBasedSchemaMatcher


class TrainSetAgent:
    """LLM-driven agent that creates blocking evaluation test sets using percentile-based sampling."""
    
    BLOCKER_TYPES = {
        "token": "TokenBlocker - blocks on shared tokens in a text column",
        "standard": "StandardBlocker - blocks on exact equality of column values",
        "sorted_neighbourhood": "SortedNeighbourhoodBlocker - sliding window over sorted key",
        "embedding": "EmbeddingBlocker - semantic similarity via embeddings"
    }
    
    SIMILARITY_FUNCTIONS = {
        "jaro_winkler": "String similarity using Jaro-Winkler (good for names, titles) - returns 0-1",
        "levenshtein": "Edit distance normalized to 0-1 (good for short strings, codes)",
        "exact_match": "Returns 1 if exactly equal (case-insensitive), 0 otherwise",
        "token_overlap": "Jaccard similarity of word tokens (good for long text with shared words)",
        "numeric_diff": "1 - |a-b|/max(a,b) for numbers (good for years, durations, prices)",
        "contains": "Returns 1 if one string contains the other, 0 otherwise"
    }

    country_map = {
                'uk': 'united kingdom', 'usa': 'united states', 'us': 'united states',
                'united kingdom of great britain and northern ireland': 'united kingdom',
                'united states of america': 'united states', 'america': 'united states',
                'deutschland': 'germany', 'de': 'germany', 'ger': 'germany',
                'france': 'france', 'fr': 'france', 'fra': 'france',
                'espana': 'spain', 'spain': 'spain', 'es': 'spain',
                'italia': 'italy', 'italy': 'italy', 'it': 'italy',
                'nederland': 'netherlands', 'netherlands': 'netherlands', 'nl': 'netherlands', 'holland': 'netherlands',
                'belgique': 'belgium', 'belgie': 'belgium', 'belgium': 'belgium', 'be': 'belgium',
                'schweiz': 'switzerland', 'suisse': 'switzerland', 'switzerland': 'switzerland', 'ch': 'switzerland',
                'osterreich': 'austria', 'austria': 'austria', 'at': 'austria',
                'canada': 'canada', 'ca': 'canada', 'can': 'canada',
                'australia': 'australia', 'au': 'australia', 'aus': 'australia',
                'japan': 'japan', 'jp': 'japan', 'jpn': 'japan', 'nippon': 'japan',
                'brasil': 'brazil', 'brazil': 'brazil', 'br': 'brazil',
                'mexico': 'mexico', 'mx': 'mexico', 'mejico': 'mexico',
                'russia': 'russia', 'ru': 'russia', 'russian federation': 'russia',
                'china': 'china', 'cn': 'china', 'prc': 'china',
                'korea': 'south korea', 'south korea': 'south korea', 'kr': 'south korea', 'republic of korea': 'south korea',
                'sweden': 'sweden', 'se': 'sweden', 'sverige': 'sweden',
                'norway': 'norway', 'no': 'norway', 'norge': 'norway',
                'denmark': 'denmark', 'dk': 'denmark', 'danmark': 'denmark',
                'finland': 'finland', 'fi': 'finland', 'suomi': 'finland',
                'poland': 'poland', 'pl': 'poland', 'polska': 'poland',
                'portugal': 'portugal', 'pt': 'portugal',
                'greece': 'greece', 'gr': 'greece', 'hellas': 'greece',
                'ireland': 'ireland', 'ie': 'ireland', 'eire': 'ireland',
                'new zealand': 'new zealand', 'nz': 'new zealand',
                'south africa': 'south africa', 'za': 'south africa',
                'argentina': 'argentina', 'ar': 'argentina',
                'india': 'india', 'in': 'india',
                'czech republic': 'czech republic', 'cz': 'czech republic', 'czechia': 'czech republic',
                'hungary': 'hungary', 'hu': 'hungary', 'magyarorszag': 'hungary',
                'romania': 'romania', 'ro': 'romania',
                'turkey': 'turkey', 'tr': 'turkey', 'turkiye': 'turkey',
                'israel': 'israel', 'il': 'israel',
                'uae': 'united arab emirates', 'united arab emirates': 'united arab emirates',
                'singapore': 'singapore', 'sg': 'singapore',
                'hong kong': 'hong kong', 'hk': 'hong kong',
                'taiwan': 'taiwan', 'tw': 'taiwan',
                'indonesia': 'indonesia', 'id': 'indonesia',
                'thailand': 'thailand', 'th': 'thailand',
                'philippines': 'philippines', 'ph': 'philippines',
                'malaysia': 'malaysia', 'my': 'malaysia',
                'vietnam': 'vietnam', 'vn': 'vietnam',
                'colombia': 'colombia', 'co': 'colombia',
                'chile': 'chile', 'cl': 'chile',
                'peru': 'peru', 'pe': 'peru',
                'ukraine': 'ukraine', 'ua': 'ukraine',
                'belarus': 'belarus', 'by': 'belarus',
                'croatia': 'croatia', 'hr': 'croatia', 'hrvatska': 'croatia',
                'serbia': 'serbia', 'rs': 'serbia',
                'slovenia': 'slovenia', 'si': 'slovenia',
                'slovakia': 'slovakia', 'sk': 'slovakia',
                'bulgaria': 'bulgaria', 'bg': 'bulgaria',
                'lithuania': 'lithuania', 'lt': 'lithuania',
                'latvia': 'latvia', 'lv': 'latvia',
                'estonia': 'estonia', 'ee': 'estonia',
            }
    
    def __init__(self, llm, output_dir: str = "output/testset-generation", n_samples: int = 100,
                 checkpoint_interval: int = 10, verbose: bool = True, target_match_pct: float = 0.20,
                 max_candidates: int = 500000, min_token_len: int = 3, confidence_threshold: float = 0.5,
                 batch_size: int = 20, target_nonmatch_rate: float = 0.6, target_match_rate: float = 0.4):
        self.llm, self.output_dir, self.n_samples = llm, output_dir, n_samples
        self.checkpoint_interval, self.verbose = checkpoint_interval, verbose
        self.target_match_pct, self.max_candidates, self.min_token_len = target_match_pct, max_candidates, min_token_len
        self.confidence_threshold = confidence_threshold
        self.batch_size = batch_size
        self.target_nonmatch_rate, self.target_match_rate = target_nonmatch_rate, target_match_rate
        # Internal state
        self._schema_analysis, self._schema_correspondences = None, None
        self._all_candidates_df, self._df_a, self._df_b = None, None, None
        self._id_col_a, self._id_col_b, self._all_candidate_ids = None, None, None
        self._blocker_instance = None
        self.token_usage = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'total_cost': 0.0}
        os.makedirs(self.output_dir, exist_ok=True)
    
    # ==================== SCHEMA & PREPROCESSING ====================
    
    def _perform_schema_matching(self, df_a: pd.DataFrame, df_b: pd.DataFrame, pair_name: str) -> List[Dict[str, Any]]:
        if self.verbose: print(f"\n   Performing schema matching with LLMBasedSchemaMatcher...")
        df_a_copy, df_b_copy = df_a.copy(), df_b.copy()
        df_a_copy.attrs["dataset_name"], df_b_copy.attrs["dataset_name"] = f"{pair_name}_a", f"{pair_name}_b"
        try:
            matcher = LLMBasedSchemaMatcher(chat_model=self.llm, num_rows=10)
            mapping_df = matcher.match(df_a_copy, df_b_copy)
            correspondences = [{"col_a": row.get('source_column'), "col_b": row.get('target_column'), "score": float(row.get('score', 0.95))}
                              for _, row in mapping_df.iterrows() if pd.notna(row.get('target_column'))]
            if self.verbose:
                print(f"   Found {len(correspondences)} column correspondences:")
                for c in correspondences: print(f"      {c['col_a']} <-> {c['col_b']} (score={c['score']:.2f})")
            return correspondences
        except Exception as e:
            if self.verbose: print(f"   Schema matching failed: {e}, falling back to same-name matching...")
            return [{"col_a": c, "col_b": c, "score": 1.0} for c in set(df_a.columns) & set(df_b.columns) if c not in ['id', '_id']]
    
    def _analyze_schema_with_llm(self, df_a: pd.DataFrame, df_b: pd.DataFrame, 
                                  schema_correspondences: List[Dict[str, Any]], sample_rows: int = 3) -> Dict[str, Any]:
        def get_column_info(df, name):
            info = [f"  - {col} (dtype={df[col].dtype}, unique={df[col].nunique()}): [{', '.join(str(v)[:50] for v in df[col].dropna().head(sample_rows))}]"
                    for col in df.columns]
            return f"{name} columns:\n" + "\n".join(info)
        
        system_prompt = """You are an expert data engineer specializing in entity matching.
Column correspondences have ALREADY been discovered. NEVER use ID columns for blocking.
Respond with reasoning followed by JSON configuration."""
        
        human_prompt = f"""Analyze these datasets:
{get_column_info(df_a, "Dataset A")}
{get_column_info(df_b, "Dataset B")}

CORRESPONDENCES:

{chr(10).join(f"  - A.{c['col_a']} <-> B.{c['col_b']} (score={c['score']:.2f})" for c in schema_correspondences)}
RULES FOR SIMILARITY COLUMNS (MUST FOLLOW):
- ALWAYS include the PRIMARY identifying columns (name/title AND artist/author/creator/Platform) in similarity_columns, even if used for blocking.
- The blocking column filters candidates, but similarity_columns VERIFY the match. Always include both.
- Use different weights: the 2-3 most important columns should have weights 0.25-0.35, supporting columns 0.10-0.20.
- Total weights should sum to approximately 1.0.
- For names/titles/artists (short identity strings): use "jaro_winkler" or "name_reorder" (handles "Last, First" vs "First Last").
- For countries: use "country" (handles "UK" vs "United Kingdom" etc).
- For dates: use "date_diff" (handles different date formats, compares years).
- For numeric fields (duration, tracks, price): use "numeric_diff".
- Do NOT use "token_overlap" for short fields - only for long descriptions.
- Do NOT use "exact_match" unless values are guaranteed to be normalized.Use this for short codes like platform

RULES FOR BLOCKING STRATEGY (MUST FOLLOW):
- Use "token" blocker on name/title columns (allows partial matches).
- Do NOT use "standard" blocker on name columns (misses fuzzy matches).
- Do NOT block on countries (format varies too much).
RESPOND WITH ONLY THE FOLLOWING JSON FORMAT (NO ADDITIONAL TEXT):
IMPORTANT: Use only the column name (e.g., "name"), NOT with prefixes like "A.name" or "B.name".
Provide JSON:
{{"entity_type": "...", "blocking_column_pair": {{"col_a": "...", "col_b": "..."}}, "blocker_type": "token|standard|embedding",
"blocker_config": {{"reason": "..."}}, "match_criteria": "...",
"similarity_columns": [{{"col_a": "...", "col_b": "...", "function": "...", "weight": 0.X}}],
"geo_config": {{"enabled": false}}, "bucket_thresholds": {{"high": 0.X, "mid": 0.X}}}}"""
        
        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            text = (response.content if hasattr(response, 'content') else str(response)).strip()
            json_start, json_end = text.find('{'), text.rfind('}') + 1
            analysis = json.loads(text[json_start:json_end]) if json_start != -1 else {}
            
            # Validate blocking column
            BAD_COLS = {'country', 'state', 'category', 'type', 'id', '_id', 'record_id', 'index'}
            blocking_pair = analysis.get('blocking_column_pair', {})
            if blocking_pair.get('col_a', '').lower() in BAD_COLS or blocking_pair.get('col_b', '').lower() in BAD_COLS:
                name_cols = [c for c in schema_correspondences if 'name' in c['col_a'].lower() and c['col_a'].lower() not in BAD_COLS]
                text_cols = [c for c in schema_correspondences if df_a[c['col_a']].dtype == 'object' and c['col_a'].lower() not in BAD_COLS]
                best = name_cols[0] if name_cols else (text_cols[0] if text_cols else schema_correspondences[0])
                analysis['blocking_column_pair'] = {'col_a': best['col_a'], 'col_b': best['col_b']}
                analysis['blocker_type'] = 'token'
            
            if self.verbose:
                print(f"   Entity type: {analysis.get('entity_type', 'unknown')}")
                print(f"   Blocker: {analysis.get('blocker_type', 'token')} on {analysis.get('blocking_column_pair', {}).get('col_a')}")
            analysis['schema_correspondences'] = schema_correspondences
            return analysis
        except Exception as e:
            if self.verbose: print(f"   LLM analysis failed: {e}")
            text_corrs = [c for c in schema_correspondences if df_a[c['col_a']].dtype == 'object' and c['col_a'] not in ['id', '_id']][:3]
            blocking_corr = text_corrs[0] if text_corrs else schema_correspondences[0]
            return {"entity_type": "unknown", "blocking_column_pair": {"col_a": blocking_corr['col_a'], "col_b": blocking_corr['col_b']},
                    "blocker_type": "token", "similarity_columns": [{"col_a": c['col_a'], "col_b": c['col_b'], "function": "jaro_winkler", "weight": 1.0/len(text_corrs)} for c in text_corrs],
                    "geo_config": {"enabled": False}, "bucket_thresholds": {"high": 0.85, "mid": 0.65}, "schema_correspondences": schema_correspondences}
    
    def _normalize_text(self, s) -> str:
        if s is None:
            return ""
        if isinstance(s, float) and pd.isna(s):
            return ""
        if isinstance(s, (pd._libs.missing.NAType,)):
            return ""
        s = str(s).strip().casefold()
        if s in {"nan", "none", "null", "<na>"}:
            return ""
        return s

    
    def _compute_similarity(self, val_a, val_b, method: str = "jaro_winkler", **kwargs) -> float:
        a = self._normalize_text(val_a)
        b = self._normalize_text(val_b)
        if not a or not b:
            return np.nan
        
        if method == "exact_match":
            return 1.0 if a == b else 0.0
        
        if method == "contains":
            if len(a) < 3 or len(b) < 3:
                return 0.0
            return 1.0 if (a in b or b in a) else 0.0
        
        if method == "numeric_diff":
            def to_float(x):
                if x is None or (isinstance(x, float) and np.isnan(x)):
                    raise ValueError("missing")
                s = str(x).strip()
                if not s or s == '0':
                    raise ValueError("missing")
                s = re.sub(r"[^0-9.\-+eE]", "", s.replace(",", ""))
                f = float(s)
                if f == 0:
                    raise ValueError("missing")
                return f
            try:
                a_num, b_num = to_float(val_a), to_float(val_b)
                denom = max(abs(a_num), abs(b_num), 1e-10)
                return float(max(0.0, min(1.0, 1.0 - abs(a_num - b_num) / denom)))
            except:
                return np.nan
        
        if method == "date_diff":
            def parse_date(val):
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    return None
                s = str(val).strip()
                if not s or s.lower() in ('nan', 'none', 'null', '0'):
                    return None
                if re.match(r'^\d{4}$', s):
                    return int(s)
                for fmt in ['%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y', '%d/%m/%Y', '%Y', '%m/%d/%Y', '%d.%m.%Y']:
                    try:
                        dt = datetime.strptime(s, fmt)
                        return dt.year + dt.month/12 + dt.day/365
                    except:
                        continue
                year_match = re.search(r'(\d{4})', s)
                return int(year_match.group(1)) if year_match else None
            try:
                date_a, date_b = parse_date(val_a), parse_date(val_b)
                if date_a is None or date_b is None:
                    return np.nan
                return float(max(0.0, 1.0 - abs(date_a - date_b) * 0.2))
            except:
                return np.nan
        
        if method == "token_overlap":
            ta, tb = set(re.findall(r"[a-z0-9]+", a)), set(re.findall(r"[a-z0-9]+", b))
            if not ta or not tb:
                return 0.0
            return float(len(ta & tb) / len(ta | tb))
        
        if method == "country":
            country_map = self.country_map
            a_norm = country_map.get(a, a)
            b_norm = country_map.get(b, b)
            if a_norm == b_norm:
                return 1.0
            if a_norm in b_norm or b_norm in a_norm:
                return 0.9
            return 0.0
        
        if method == "name_reorder":
            def normalize_name(s):
                s = re.sub(r'\s*\([^)]*\)', '', s)
                s = re.sub(r'\s*feat\.?.*$', '', s, flags=re.IGNORECASE)
                parts = [p.strip() for p in re.split(r'[,|]', s) if p.strip()]
                parts = sorted([p.lower() for p in parts])
                return ' '.join(parts)
            a_norm, b_norm = normalize_name(a), normalize_name(b)
            if a_norm == b_norm:
                return 1.0
            return float(textdistance.jaro_winkler(a_norm, b_norm))
        
        if method == "levenshtein":
            return 1.0 if a == b else float(Levenshtein.normalized_similarity(a, b))
        
        return float(textdistance.jaro_winkler(a, b))

            
    def _haversine_km(self, lat1, lon1, lat2, lon2) -> float:
        try:
            if None in (lat1, lon1, lat2, lon2) or any(pd.isna(x) for x in [lat1, lon1, lat2, lon2]): return float("inf")
            rlat1, rlon1, rlat2, rlon2 = map(math.radians, map(float, [lat1, lon1, lat2, lon2]))
            dlat, dlon = rlat2 - rlat1, rlon2 - rlon1
            a = math.sin(dlat/2)**2 + math.cos(rlat1)*math.cos(rlat2)*math.sin(dlon/2)**2
            return 6371.0088 * (2 * math.asin(math.sqrt(a)))
        except: return float("inf")
    
    def _compute_geo_similarity(self, lat1, lon1, lat2, lon2, threshold_m: float = 100) -> float:
        dist_km = self._haversine_km(lat1, lon1, lat2, lon2)
        if dist_km == float('inf'): return 0.0
        dist_m = dist_km * 1000
        if dist_m <= threshold_m: return 1.0
        max_dist = threshold_m * 10
        return max(0, 1 - (dist_m - threshold_m) / (max_dist - threshold_m)) if dist_m < max_dist else 0.0
    
    # ==================== BLOCKER & CANDIDATES ====================
    
    def _create_pydi_blocker(self, df_a: pd.DataFrame, df_b: pd.DataFrame, id_col_a: str, id_col_b: str, schema_analysis: Dict):
        blocker_type = schema_analysis.get('blocker_type', 'token')
        blocking_pair = schema_analysis.get('blocking_column_pair', {})
        schema_correspondences = schema_analysis.get('schema_correspondences', [])
        
        df_left, df_right = df_a.copy(), df_b.copy()
        df_left['_id'], df_right['_id'] = df_left[id_col_a].astype(str), df_right[id_col_b].astype(str)
        
        rename_map = {c['col_b']: c['col_a'] for c in schema_correspondences if c['col_a'] != c['col_b'] and c['col_b'] in df_right.columns}
        if rename_map and self.verbose: print(f"   Renaming columns in Dataset B: {rename_map}")
        df_right = df_right.rename(columns=rename_map)
        
        block_col = blocking_pair.get('col_a')
        if self.verbose: print(f"   Creating PyDI {blocker_type} blocker on column '{block_col}'...")
        
        try:
            common = set(df_left.columns) & set(df_right.columns) - {'_id'}
            text_cols = [c for c in common if df_left[c].dtype == 'object']
            if block_col is None or block_col not in df_left.columns: block_col = text_cols[0] if text_cols else list(common)[0]
            
            if blocker_type == "token":
                blocker = TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column='_id',
                                       min_token_len=self.min_token_len, preprocess=self._normalize_text)
            elif blocker_type == "standard":
                blocker = StandardBlocker(df_left=df_left, df_right=df_right, on=[block_col], id_column='_id', preprocess=self._normalize_text)
            elif blocker_type == "sorted_neighbourhood":
                blocker = SortedNeighbourhoodBlocker(df_left=df_left, df_right=df_right, key=block_col, id_column='_id',
                                                    window=schema_analysis.get('blocker_config', {}).get('window_size', 5), preprocess=self._normalize_text)
            elif blocker_type == "embedding":
                blocker = EmbeddingBlocker(df_left=df_left, df_right=df_right, text_cols=[block_col] if block_col else text_cols[:1],
                                          id_column='_id', model='sentence-transformers/all-MiniLM-L6-v2', top_k=50, threshold=0.3)
            else:
                blocker = TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column='_id',
                                       min_token_len=self.min_token_len, preprocess=self._normalize_text)
            return blocker, df_left, df_right, '_id', rename_map
        except Exception as e:
            if self.verbose: print(f"   Failed to create {blocker_type}: {e}, falling back to TokenBlocker")
            block_col = text_cols[0] if text_cols else list(common)[0]
            return TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column='_id',
                               min_token_len=self.min_token_len, preprocess=self._normalize_text), df_left, df_right, '_id', rename_map
    
    def _build_candidates(self, df_a: pd.DataFrame, df_b: pd.DataFrame, id_col_a: str, id_col_b: str, schema_analysis: Dict) -> pd.DataFrame:
        similarity_columns = schema_analysis.get('similarity_columns', [])
        geo_config = schema_analysis.get('geo_config', {'enabled': False})
        
        blocker, df_left, df_right, id_column, rename_map = self._create_pydi_blocker(df_a, df_b, id_col_a, id_col_b, schema_analysis)
        self._blocker_instance = blocker
        
        try:
            candidate_pairs = blocker.materialize()
            if self.verbose: print(f"   Materialized {len(candidate_pairs):,} candidate pairs")
            if len(candidate_pairs) > self.max_candidates:
                print(f"    TRUNCATING: {len(candidate_pairs):,} -> {self.max_candidates:,}")
                candidate_pairs = candidate_pairs.sample(n=self.max_candidates, random_state=42)
        except Exception as e:
            if self.verbose: print(f"   Blocker materialization failed: {e}")
            return pd.DataFrame()
        
        if candidate_pairs.empty: return pd.DataFrame()
        
        cand = candidate_pairs.rename(columns={'id1': 'id_a', 'id2': 'id_b'})
        self._all_candidate_ids = set(zip(cand['id_a'].astype(str), cand['id_b'].astype(str)))
        
        df_left_idx, df_right_idx = df_left.set_index('_id'), df_right.set_index('_id')
        for col in df_a.columns:
            if col != id_col_a: cand[f'{col}_a'] = cand['id_a'].map(lambda x: df_left_idx.loc[x, col] if x in df_left_idx.index else None)
        for col in df_b.columns:
            if col != id_col_b:
                lookup = rename_map.get(col, col)
                if lookup in df_right_idx.columns: cand[f'{col}_b'] = cand['id_b'].map(lambda x, lc=lookup: df_right_idx.loc[x, lc] if x in df_right_idx.index else None)
        
        if self.verbose and similarity_columns: print(f"   Computing similarity scores for {len(similarity_columns)} column pairs...")
        
        weighted_scores, total_weight = [], 0
        for sim in similarity_columns:
            col_a, col_b = sim.get('col_a') or sim.get('column'), sim.get('col_b') or sim.get('column')
            func, weight = sim.get('function', 'jaro_winkler'), sim.get('weight', 1.0)
            col_a_full, col_b_full = f'{col_a}_a', f'{col_b}_b'
            if col_a_full in cand.columns and col_b_full in cand.columns:
                sim_col = f"sim__{col_a}__{col_b}__{func}"
                cand[sim_col] = cand.apply(
                    lambda r, ca=col_a_full, cb=col_b_full, f=func: self._compute_similarity(r.get(ca), r.get(cb), method=f),axis=1)
                weighted_scores.append((sim_col, weight))
                total_weight += weight

            else:
                 print(f"   SKIP sim: missing columns {col_a_full} or {col_b_full}")

        if self.verbose:
            print(f"   Weighted scores computed: {weighted_scores}")
            for col, w in weighted_scores:
                print(f"      {col}: min={cand[col].min():.3f}, max={cand[col].max():.3f}, mean={cand[col].mean():.3f}")
        
        if geo_config.get('enabled', False):
            lat_a, lon_a = f"{geo_config.get('lat_a', 'latitude')}_a", f"{geo_config.get('lon_a', 'longitude')}_a"
            lat_b, lon_b = f"{geo_config.get('lat_b', 'latitude')}_b", f"{geo_config.get('lon_b', 'longitude')}_b"
            if all(c in cand.columns for c in [lat_a, lon_a, lat_b, lon_b]):
                threshold_m, geo_weight = geo_config.get('threshold_meters', 100), geo_config.get('weight', 0.3)
                cand['sim_geo'] = cand.apply(lambda r: self._compute_geo_similarity(r.get(lat_a), r.get(lon_a), r.get(lat_b), r.get(lon_b), threshold_m), axis=1)
                weighted_scores.append(('sim_geo', geo_weight))
                total_weight += geo_weight
        def compute_row_score(row):
            valid_sims = []
            for col, weight in weighted_scores:
                val = row.get(col)
                if val is not None and not pd.isna(val) and val > 0.01:
                    valid_sims.append((val, weight))
            
            if len(valid_sims) < 2:
                return 0.0
            
            total_weight = sum(w for _, w in valid_sims)
            max_weight = max(w for _, w in valid_sims)
            
            # Separate high-weight (key) vs low-weight (supporting) columns
            # Key = weight >= 50% of max weight
            key_sims = [(v, w) for v, w in valid_sims if w >= max_weight * 0.5]
            support_sims = [(v, w) for v, w in valid_sims if w < max_weight * 0.5]
            
            # Key columns: geometric mean (multiplicative - low values hurt a lot)
            if key_sims:
                key_product = 1.0
                for v, w in key_sims:
                    key_product *= v ** w
                key_score = key_product ** (1.0 / sum(w for _, w in key_sims))
            else:
                key_score = 0.0
            
            # Supporting columns: arithmetic mean (additive - more lenient)
            if support_sims:
                support_score = sum(v * w for v, w in support_sims) / sum(w for _, w in support_sims)
            else:
                support_score = 0.5
            
            # Combine: key columns dominate
            key_total = sum(w for _, w in key_sims) if key_sims else 0
            support_total = sum(w for _, w in support_sims) if support_sims else 0
            
            if key_total + support_total == 0:
                return 0.0
    
            return (key_score * key_total + support_score * support_total) / (key_total + support_total)
        if weighted_scores:
            cand['score'] = cand.apply(compute_row_score, axis=1)
        else:
            cand['score'] = 0.0
        if self.verbose and 'score' in cand.columns:
            s = cand['score']
            print(f"   Score distribution: min={s.min():.3f}, max={s.max():.3f}, mean={s.mean():.3f}, median={s.median():.3f}")
            print(f"   Percentiles: 25th={s.quantile(0.25):.3f}, 50th={s.quantile(0.50):.3f}, 75th={s.quantile(0.75):.3f}, 90th={s.quantile(0.90):.3f}")
        
        self._all_candidates_df = cand.copy()
        high_thresh, mid_thresh = cand['score'].quantile(0.80), cand['score'].quantile(0.50)
        cand = cand.drop_duplicates(subset=['id_a', 'id_b'])
        return cand.sort_values('score', ascending=False).reset_index(drop=True)
    
    # ==================== LABELING ====================
    
    def _format_record_for_prompt(self, row: pd.Series, suffix: str) -> str:
        exclude = ['_block', 'sim_', 'score', 'bucket', 'label', 'geo_km', 'geo_m', 'sample_zone', 'rolling_', 'dist_to_', 'cumulative_', 'bin_p_']
        lines = [f"  {col.replace(f'_{suffix}', '')}: {row[col]}" for col in sorted(row.index)
                 if col.endswith(f"_{suffix}") and not any(col.startswith(p) for p in exclude) and pd.notna(row[col]) and str(row[col]).strip()]
        return "\n".join(lines) if lines else "  (no attributes available)"
    
    def _label_batch(self, batch_df: pd.DataFrame, schema_analysis: Dict) -> List[int]:
        entity_type = schema_analysis.get('entity_type', 'entity')
        system_prompt = f"""
You are an expert entity resolver. Decide if two {entity_type} records refer to the same real-world entity.
Return a JSON array with objects: {{"match": true|false, "confidence": <float 0.0-1.0>}}

RULES:
1. PRIMARY IDENTIFIERS (name/title + artist/creator/author): Must match (case-insensitive). Both matching = likely MATCH.
    -The match must not be excact, but very similar (typos, word order, minor differences allowed).
    - For example 'Live at Spirit Square' by 'Jones, Marti' vs  '[explicit] Live at Spirit Square' by 'M. Jones' is still a MATCH.

2. HARD CONFLICTS (any = NON-MATCH):
   - Platform: PC vs PS4 vs Xbox vs Switch = DIFFERENT entities
   - Legal suffix: "Chevron" vs "Chevron Inc" vs "Chevron Corp" vs "Chevron Solutions" = DIFFERENT entities
   - Country: If clearly different (allow "UK"="United Kingdom"="GB")
   - Year: If clearly different (allow "2005"="2005-01-01")
   - Duration: If difference >30% and both >0 = DIFFERENT (e.g., 200s vs 400s = non-match)

3. IGNORE THESE (not conflicts):
   - Duration difference <30% (measurement variance)
   - Duration=0 means MISSING, not zero seconds
   - Ratings, prices, sales (volatile data)
   - Missing values

4. SUMMARY:
   - Same name+artist, similar duration (<30% diff) → MATCH
   - Same name+artist, very different duration (>30% diff) → NON-MATCH
   - Same name, different artist → NON-MATCH
   - Different platform or legal suffix → NON-MATCH

confidence: 1.0=definitely same, 0.0=definitely different. match=true if confidence>={self.confidence_threshold}
Respond with ONLY the JSON array."""
        
        pairs = [f"--- PAIR {i+1} ---\nRecord A:\n{self._format_record_for_prompt(row, 'a')}\nRecord B:\n{self._format_record_for_prompt(row, 'b')}\n"
                 for i, (_, row) in enumerate(batch_df.iterrows())]
        human_prompt = f"Decide if these {len(batch_df)} record pairs refer to the same entity:\n\n{''.join(pairs)}"
        
        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            text = (response.content if hasattr(response, 'content') else str(response)).strip()
            if text.startswith("```"): text = "\n".join(l for l in text.split("\n") if not l.strip().startswith("```"))
            results = json.loads(text)
            if isinstance(results, list) and len(results) == len(batch_df):
                labels = []
                for r in results:
                    if 'confidence' in r:
                        labels.append(1 if r['confidence'] >= self.confidence_threshold else 0)
                    else:
                        labels.append(1 if r.get('match', False) else 0)
                return labels
        except: pass
        # Fallback: label one by one
        return [self._label_single(row, i, len(batch_df), schema_analysis) for i, (_, row) in enumerate(batch_df.iterrows())]
    
    def _label_single(self, row: pd.Series, idx: int, total: int, schema_analysis: Dict) -> int:
        entity_type = schema_analysis.get('entity_type', 'entity')
        system_prompt = f"You are an expert at matching {entity_type} records. Label 1 for SAME entity, 0 otherwise. RESPOND WITH ONLY: 1 or 0"
        human_prompt = f"RECORD A:\n{self._format_record_for_prompt(row, 'a')}\n\nRECORD B:\n{self._format_record_for_prompt(row, 'b')}\n\nSimilarity: {row.get('score', 0):.2f}\n\nSame {entity_type}? Reply 1 or 0:"
        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            return 1 if '1' in (response.content if hasattr(response, 'content') else str(response)).strip()[:5] else 0
        except: return 0
    
    # ==================== BIN-HOPPING STRATEGY ====================
    
    def _create_percentile_bins(self, scores: pd.Series) -> List[Dict]:
        """Create percentile bins: fine at top (p99, p98...), coarse at bottom (0-20, 20-40...)."""
        # Fine bins at top where matches are sparse
        fine_percentiles = [1.00,0.999, 0.99, 0.98, 0.97, 0.96, 0.95, 0.93, 0.90, 0.85, 0.80]
        # Coarse bins at bottom
        coarse_percentiles = [0.60, 0.40,0.0]
        
        all_percentiles = sorted(set(fine_percentiles + coarse_percentiles), reverse=True)
        bins = []
        for i in range(len(all_percentiles) - 1):
            p_high, p_low = all_percentiles[i], all_percentiles[i + 1]
            score_high, score_low = scores.quantile(p_high), scores.quantile(p_low)
            bins.append({'p_high': p_high, 'p_low': p_low, 'score_high': score_high, 'score_low': score_low})
        
        if self.verbose:
            print(f"\n   Created {len(bins)} percentile bins:")
            for b in bins[:5]: print(f"      p{int(b['p_low']*100)}-p{int(b['p_high']*100)}: score {b['score_low']:.3f}-{b['score_high']:.3f}")
            if len(bins) > 5: print(f"      ... and {len(bins)-5} more")
        return bins
    
    def _get_bin_candidates(self, candidates: pd.DataFrame, bin_info: Dict, labeled_pairs: set) -> pd.DataFrame:
        """Get unlabeled candidates from a specific bin."""
        df = candidates[(candidates['score'] >= bin_info['score_low']) & (candidates['score'] <= bin_info['score_high'])].copy()
        df = df[~df.apply(lambda r: (str(r['id_a']), str(r['id_b'])) in labeled_pairs, axis=1)]
        return df
    
    def _build_dataset_binhopping(self, candidates: pd.DataFrame, schema_analysis: Dict, n_matches_needed: int, seed: int = 42) -> pd.DataFrame:
        """Build dataset by hopping between percentile bins based on target rates."""
        np.random.seed(seed)
        n_non_matches_needed = self.n_samples - n_matches_needed
        
        scores = candidates['score']
        bins = self._create_percentile_bins(scores)
        
        labeled_pairs = set()
        selected_matches, selected_non_matches = [], []
        
        # PHASE 1: Collect non-matches starting from p100 (highest similarity)
        if self.verbose:
            print(f"\n   PHASE 1: Collecting non-matches ({n_non_matches_needed} needed)")
            print(f"      Target non-match rate: {self.target_nonmatch_rate:.0%}")
            print(f"      Starting from TOP (p100) and moving DOWN as needed")
        
        bin_idx = 0  # Start at top bin
        while len(selected_non_matches) < n_non_matches_needed:
            current_bin = bins[bin_idx]
            pool = self._get_bin_candidates(candidates, current_bin, labeled_pairs)
            
            if pool.empty:
                bin_idx += 1  # Move down
                continue
            
            # Sample a batch from this bin
            batch = pool.sample(n=min(self.batch_size, len(pool)), random_state=seed + bin_idx).copy()
            batch['label'] = self._label_batch(batch, schema_analysis)
            
            batch_nonmatches = 0
            for _, r in batch.iterrows():
                labeled_pairs.add((str(r['id_a']), str(r['id_b'])))
                if r['label'] == 0 and len(selected_non_matches) < n_non_matches_needed:
                    selected_non_matches.append(r); batch_nonmatches += 1
                # Don't collect matches here - Phase 2 will handle them independently
            
            rate = batch_nonmatches / len(batch) if len(batch) > 0 else 0
            if self.verbose:
                print(f"      p{int(current_bin['p_low']*100)}-p{int(current_bin['p_high']*100)}: {batch_nonmatches}/{len(batch)} non-matches ({rate:.0%}), total: {len(selected_non_matches)}/{n_non_matches_needed}")
            
            # Decide whether to move up or down
            if rate < self.target_nonmatch_rate:
                # Too few non-matches (too many matches here), move DOWN to lower similarity
                if self.verbose and bin_idx < len(bins) - 1: print(f"      → Below target rate, moving DOWN")
                bin_idx += 1
            elif rate > self.target_nonmatch_rate and bin_idx > 0:
                # Too many non-matches (could find harder ones), move UP to higher similarity
                if self.verbose: print(f"      → Above target rate, moving UP for harder non-matches")
                bin_idx -= 1
            # else: stay in current bin
        
        # PHASE 2: Collect matches starting from p0 (lowest similarity)
        if self.verbose:
            print(f"\n   PHASE 2: Collecting matches ({n_matches_needed - len(selected_matches)} needed)")
            print(f"      Target match rate: {self.target_match_rate:.0%}")
            print(f"      Starting from BOTTOM (p0) and moving UP as needed")
        labeled_pairs_phase2 = set()
        bin_idx = len(bins) - 1  # Start at bottom bin
        zero_match_rounds = 0
        round_seed = 0
        while len(selected_matches) < n_matches_needed:
            current_bin = bins[bin_idx]
            pool = self._get_bin_candidates(candidates, current_bin, labeled_pairs_phase2)
            
            if pool.empty:
                if bin_idx == 0:
                    zero_match_rounds += 1
                    if zero_match_rounds >= 5:
                        if self.verbose: print(f"      → No more candidates, stopping")
                        break
                    continue
                bin_idx -= 1
                continue
            
            batch = pool.sample(n=min(self.batch_size, len(pool)), random_state=seed + bin_idx + 100 + round_seed).copy()
            round_seed += 1
            batch['label'] = self._label_batch(batch, schema_analysis)
            
            batch_matches = 0
            for _, r in batch.iterrows():
                labeled_pairs_phase2.add((str(r['id_a']), str(r['id_b'])))
                if r['label'] == 1 and len(selected_matches) < n_matches_needed:
                    selected_matches.append(r)
                    batch_matches += 1
            
            rate = batch_matches / len(batch) if len(batch) > 0 else 0
            if self.verbose:
                print(f"      p{int(current_bin['p_low']*100)}-p{int(current_bin['p_high']*100)}: {batch_matches}/{len(batch)} matches ({rate:.0%}), total: {len(selected_matches)}/{n_matches_needed}")
            
            if batch_matches == 0:
                zero_match_rounds += 1
                if bin_idx == 0:
                    if zero_match_rounds >= 5:
                        if self.verbose: print(f"      → 5 rounds with 0 matches at top bin, stopping")
                        break
                    if self.verbose: print(f"      → No matches at top bin, retrying ({zero_match_rounds}/5)")
                else:
                    bin_idx -= 1
                    if self.verbose: print(f"      → No matches, moving UP")
            else:
                zero_match_rounds = 0
                if bin_idx == 0:
                    if self.verbose: print(f"      → Found matches at top bin, continuing")
                elif rate < self.target_match_rate:
                    bin_idx -= 1
                    if self.verbose: print(f"      → Below target rate, moving UP")
                elif rate > self.target_match_rate and bin_idx < len(bins) - 1:
                    bin_idx += 1
                    if self.verbose: print(f"      → Above target rate, moving DOWN for harder matches")
        
        final_df = pd.DataFrame(selected_matches[:n_matches_needed] + selected_non_matches[:n_non_matches_needed])
        if self.verbose:
            print(f"\n   FINAL DATASET:")
            print(f"      Total: {len(final_df)}")
            print(f"      Matches: {len(selected_matches[:n_matches_needed])}")
            print(f"      Non-matches: {len(selected_non_matches[:n_non_matches_needed])}")
        return final_df.reset_index(drop=True)
    
    # ==================== OUTPUT & CACHING ====================
    
    def _build_reviewable_detailed_df(self, samples: pd.DataFrame, schema_analysis: Dict) -> pd.DataFrame:
        cols = ['label', 'score', 'id_a', 'id_b']
        for sim in schema_analysis.get('similarity_columns', []):
            col_a = sim.get('col_a') or sim.get('column')
            for c in [f'{col_a}_a', f'{sim.get("col_b") or col_a}_b', f'sim_{col_a}']:
                if c in samples.columns: cols.append(c)
        for gc in ['geo_km', 'geo_m', 'sim_geo']:
            if gc in samples.columns: cols.append(gc)
        cols = [c for c in cols if c in samples.columns]
        detailed = samples[cols].copy()
        for col in detailed.columns:
            if col.startswith('sim_') or col == 'score': detailed[col] = detailed[col].round(3)
        return detailed
    
    def _get_checkpoint_path(self, pair_name: str) -> str: return os.path.join(self.output_dir, f"checkpoint_{pair_name}.csv")
    def _get_schema_cache_path(self, pair_name: str) -> str: return os.path.join(self.output_dir, f"schema_{pair_name}.json")
    
    def _load_checkpoint(self, pair_name: str) -> Tuple[Optional[pd.DataFrame], int]:
        path = self._get_checkpoint_path(pair_name)
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                labeled = df['label'].notna().sum()
                if self.verbose: print(f"Found checkpoint: {labeled} samples labeled")
                return df, int(labeled)
            except: pass
        return None, 0
    
    def _load_schema_cache(self, pair_name: str) -> Optional[Dict]:
        path = self._get_schema_cache_path(pair_name)
        if os.path.exists(path):
            try:
                with open(path, 'r') as f:
                    schema = json.load(f)
                    if 'schema_correspondences' in schema and 'similarity_columns' in schema: return schema
            except: pass
        return None
    
    def _save_schema_cache(self, schema: Dict, pair_name: str):
        try:
            with open(self._get_schema_cache_path(pair_name), 'w') as f: json.dump(schema, f, indent=2)
        except: pass
    
    def _save_checkpoint(self, df: pd.DataFrame, pair_name: str, labeled_count: int):
        df.to_csv(self._get_checkpoint_path(pair_name), index=False)
    
    # ==================== MAIN RUN ====================
    
    def run(self, df_a: pd.DataFrame, df_b: pd.DataFrame, pair_name: str, id_col_a: str, id_col_b: str) -> pd.DataFrame:
        print(f"\nTraining SET GENERATOR: {pair_name}")
        print(f"{'='*60}")
        print(f"Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)")
        print(f"   Target non-match rate: {self.target_nonmatch_rate:.0%}, Target match rate: {self.target_match_rate:.0%}")
        print(f"   Confidence threshold: {self.confidence_threshold}")
        
        self._df_a, self._df_b = df_a, df_b
        self._id_col_a, self._id_col_b = id_col_a, id_col_b
        self.token_usage = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'total_cost': 0.0}
        
        with get_openai_callback() as cb:
            result = self._run_internal(pair_name)
            self.token_usage = {'prompt_tokens': cb.prompt_tokens, 'completion_tokens': cb.completion_tokens, 'total_tokens': cb.total_tokens, 'total_cost': cb.total_cost}
            print(f"\nTOKEN USAGE:")
            print(f"   Prompt tokens: {cb.prompt_tokens:,}")
            print(f"   Completion tokens: {cb.completion_tokens:,}")
            print(f"   Total tokens: {cb.total_tokens:,}")
            print(f"   Estimated cost: ${cb.total_cost:.4f}")
        return result
    
    def _run_internal(self, pair_name: str) -> pd.DataFrame:
        checkpoint_df, start_idx = self._load_checkpoint(pair_name)
        if checkpoint_df is not None and start_idx >= self.n_samples:
            print(f"Already complete!")
            return checkpoint_df[['id_a', 'id_b', 'label']]
        
        schema_analysis = self._load_schema_cache(pair_name)
        if schema_analysis is None:
            schema_correspondences = self._perform_schema_matching(self._df_a, self._df_b, pair_name)
            self._schema_correspondences = schema_correspondences
            schema_analysis = self._analyze_schema_with_llm(self._df_a, self._df_b, schema_correspondences)
            self._save_schema_cache(schema_analysis, pair_name)
        else:
            self._schema_correspondences = schema_analysis.get('schema_correspondences', [])
        
        self._schema_analysis = schema_analysis
        schema_analysis['_pair_name'] = pair_name
        
        if checkpoint_df is None:
            candidates = self._build_candidates(self._df_a, self._df_b, self._id_col_a, self._id_col_b, schema_analysis)
            if candidates.empty:
                print("No candidates found!")
                return pd.DataFrame(columns=['id_a', 'id_b', 'label'])
            
            n_matches_needed = int(self.n_samples * self.target_match_pct)
            samples = self._build_dataset_binhopping(candidates, schema_analysis, n_matches_needed)
        else:
            samples = checkpoint_df
        
        # Save outputs
        output_path = os.path.join(self.output_dir, f"{pair_name}_goldstandard_blocking.csv")
        gold_standard = samples[['id_a', 'id_b', 'label']].copy()
        gold_standard['label'] = gold_standard['label'].astype(int)
        gold_standard.to_csv(output_path, index=False)
        detailed_df = self._build_reviewable_detailed_df(samples, schema_analysis)
        detailed_df.to_csv(os.path.join(self.output_dir, f"{pair_name}_detailed.csv"), index=False)
        
        # Cleanup checkpoint
        path = self._get_checkpoint_path(pair_name)
        if os.path.exists(path): os.remove(path)
        
        return gold_standard


def get_llm(provider: str = "gpt"):
    if provider == "gpt":
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    elif provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
    elif provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
    else:
        raise ValueError(f"Unknown provider: {provider}")

MUSIC DATASETS

In [10]:
# Music datasets discogs and musicbrainz
import pandas as pd
discogs = pd.read_xml("input/datasets/discogs.xml")
musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")
lastfm = pd.read_xml("input/datasets/lastfm.xml")

llm = get_llm("gpt")

trainset_agent_music = TrainSetAgent(
    llm=llm,
    output_dir="output/big/music4",
    n_samples=300,              # Total samples to label
    verbose=True,
    target_match_pct=0.5,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=1000000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on discogs vs musicbrainz
gold_standard_music = trainset_agent_music.run(
    df_a=discogs,
    df_b=musicbrainz,
    pair_name="discogs_musicbrainz",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: discogs_musicbrainz
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 7 column correspondences:
      id <-> id (score=0.95)
      name <-> name (score=0.95)
      artist <-> artist (score=0.95)
      release-date <-> release-date (score=0.95)
      release-country <-> release-country (score=0.95)
      duration <-> duration (score=0.95)
      tracks <-> tracks (score=0.95)
   Entity type: music_recordings
   Blocker: token on name
   Creating PyDI token blocker on column 'name'...
   Materialized 1,143,051 candidate pairs
    TRUNCATING: 1,143,051 -> 1,000,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__artist__artist__jaro_winkler', 0.3), ('sim__release-date__release-date__date_diff', 0.15), ('sim__release-country_

In [11]:
# Music datasets musicbrainz and lastfm
import pandas as pd
discogs = pd.read_xml("input/datasets/discogs.xml")
musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")
lastfm = pd.read_xml("input/datasets/lastfm.xml")

llm = get_llm("gpt")
trainset_agent_music = TrainSetAgent(
    llm=llm,
    output_dir="output/big/music4",
    n_samples=300,              # Total samples to label
    verbose=True,
    target_match_pct=0.5,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=1000000,       # Limit candidate pairs to 1M
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on musicbrainz vs lastfm
gold_standard_music = trainset_agent_music.run(
    df_a=musicbrainz,
    df_b=lastfm,
    pair_name="musicbrainz_lastfm",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: musicbrainz_lastfm
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 5 column correspondences:
      id <-> id (score=0.95)
      name <-> name (score=0.95)
      artist <-> artist (score=0.95)
      duration <-> duration (score=0.95)
      tracks <-> tracks (score=0.95)
   Entity type: music
   Blocker: token on name
   Creating PyDI token blocker on column 'name'...
   Materialized 505,108 candidate pairs
   Computing similarity scores for 5 column pairs...
   SKIP sim: missing columns release-date_a or release-date_b
   SKIP sim: missing columns release-country_a or release-country_b
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__artist__artist__jaro_winkler', 0.3), ('sim__duration__duration__numeric_diff', 0.15)]
      sim__name__name__jaro_winkler: min=0.000, max=1.0

In [16]:
# Music datasets discogs and lastfm
import pandas as pd
discogs = pd.read_xml("input/datasets/discogs.xml")
musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")
lastfm = pd.read_xml("input/datasets/lastfm.xml")

llm = get_llm("gpt")
trainset_agent_music = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/music7",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on discogs vs lastfm
gold_standard_music = trainset_agent_music.run(
    df_a=discogs,
    df_b=lastfm,
    pair_name="discogs_lastfm",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: discogs_lastfm
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5
   Creating PyDI token blocker on column 'name'...
   Materialized 2,529,232 candidate pairs
    TRUNCATING: 2,529,232 -> 100,000
   Computing similarity scores for 5 column pairs...
   SKIP sim: missing columns release-date_a or release-date_b
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__artist__artist__jaro_winkler', 0.3), ('sim__duration__duration__numeric_diff', 0.15), ('sim__tracks__tracks__numeric_diff', 0.1)]
      sim__name__name__jaro_winkler: min=0.000, max=1.000, mean=0.613
      sim__artist__artist__jaro_winkler: min=0.000, max=1.000, mean=0.454
      sim__duration__duration__numeric_diff: min=0.003, max=1.000, mean=0.512
      sim__tracks__tracks__numeric_diff: min=nan, max=nan, mean=nan
   Score distribution: min=0.000, max=1.000, mean=0.521, median=0.534
   Pe

GAMES DATASETS

In [5]:
#test on sales and metacritic
sales= pd.read_xml('input/datasets/sales.xml')
metacritic = pd.read_xml('input/datasets/metacritic.xml')

llm = get_llm("gpt")

trainset_agent_games = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/games4",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

gold_standard_games = trainset_agent_games.run(
    df_a=sales,
    df_b=metacritic,
    pair_name="metacritic_sales",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: metacritic_sales
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 9 column correspondences:
      id <-> id (score=0.95)
      name <-> name (score=0.95)
      releaseYear <-> releaseYear (score=0.95)
      developer <-> developer (score=0.95)
      genres <-> genres (score=0.95)
      platform <-> platform (score=0.95)
      criticScore <-> criticScore (score=0.95)
      userScore <-> userScore (score=0.95)
      ESRB <-> ESRB (score=0.95)
   Entity type: video_game
   Blocker: token on name
   Creating PyDI token blocker on column 'name'...
   Materialized 908,547 candidate pairs
    TRUNCATING: 908,547 -> 100,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__platform__platform__exact_match', 0.25), ('sim__releaseYe

In [13]:
#test on dpedia_games vs sales
sales= pd.read_xml('input/datasets/sales_with_ids.xml')
dbpedia_games = pd.read_xml('input/datasets/metacritic_with_ids.xml')

llm = get_llm("gpt")

trainset_agent_games = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/games2",
    n_samples=10,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

gold_standard_games = trainset_agent_games.run(
    df_a=dbpedia_games,
    df_b=sales,
    pair_name="dbpedia_games_sales",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: dbpedia_games_sales
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 9 column correspondences:
      id <-> id (score=0.95)
      name <-> name (score=0.95)
      releaseYear <-> releaseYear (score=0.95)
      developer <-> developer (score=0.95)
      genres <-> genres (score=0.95)
      platform <-> platform (score=0.95)
      criticScore <-> criticScore (score=0.95)
      userScore <-> userScore (score=0.95)
      ESRB <-> ESRB (score=0.95)
   Entity type: video_game
   Blocker: token on name
   Creating PyDI token blocker on column 'name'...
   Materialized 908,547 candidate pairs
    TRUNCATING: 908,547 -> 100,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__platform__platform__exact_match', 0.25), ('sim__releas

In [15]:
#test on metacritic vs dpedia_games 
sales= pd.read_xml('input/datasets/sales_with_ids.xml')
dbpedia_games = pd.read_xml('input/datasets/dbpedia_games_with_ids.xml')
metacritic = pd.read_xml('input/datasets/metacritic_with_ids.xml')

llm = get_llm("gpt")

trainset_agent_games = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/games4",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

gold_standard_games = trainset_agent_games.run(
    df_a=metacritic,
    df_b=dbpedia_games,
    pair_name="metacritic_dbpedia_games",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: metacritic_dbpedia_games
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 6 column correspondences:
      id <-> id (score=0.95)
      name <-> gameLabel (score=0.95)
      releaseYear <-> releaseDate (score=0.95)
      developer <-> developer (score=0.95)
      genres <-> genre (score=0.95)
      platform <-> platform (score=0.95)
   Entity type: video_game
   Blocker: token on name
   Renaming columns in Dataset B: {'gameLabel': 'name', 'releaseDate': 'releaseYear', 'genre': 'genres'}
   Creating PyDI token blocker on column 'name'...
   Materialized 9,668,084 candidate pairs
    TRUNCATING: 9,668,084 -> 100,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__gameLabel__jaro_winkler', 0.35), ('sim__platform__platform__exact_match', 0.25), ('sim__develo

Company Datasets

In [ ]:
## company datasets forbes and fullcontact
fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
dpedia = pd.read_xml('input/datasets/dbpedia.xml')
forbes = pd.read_csv('input/datasets/forbes.csv')


llm = get_llm("gpt")

testset_agent_companies = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/companies2",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on forbes vs fullcontact
gold_standard_companies = testset_agent_companies.run(
    df_a=forbes,
    df_b=fullcontact,
    pair_name="forbes_fullcontact",
    id_col_a="id",
    id_col_b="id"
)



Training SET GENERATOR: forbes_fullcontact
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 3 column correspondences:
      name <-> name (score=0.95)
      country <-> country (score=0.95)
      id <-> id (score=0.95)
   LLM analysis failed: Expecting ',' delimiter: line 11 column 88 (char 610)
   Creating PyDI token blocker on column 'name'...
   Materialized 27,403 candidate pairs
   Computing similarity scores for 2 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.5), ('sim__country__country__jaro_winkler', 0.5)]
      sim__name__name__jaro_winkler: min=0.000, max=1.000, mean=0.619
      sim__country__country__jaro_winkler: min=0.000, max=1.000, mean=0.369
   Score distribution: min=0.000, max=1.000, mean=0.290, median=0.435
   Percentiles: 25th=0.000, 50th=0.435, 75th=0.554, 90th=0.610



In [4]:
## company datasets forbes vs dbpedia
fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
dpedia = pd.read_xml('input/datasets/dbpedia.xml')
forbes = pd.read_csv('input/datasets/forbes.csv')


llm = get_llm("gpt")

testset_agent_companies = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/companies2",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on forbes vs dpedia
gold_standard_ky = testset_agent_companies.run(
    df_a=forbes,
    df_b=dpedia,
    pair_name="forbes_dpedia",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: forbes_dpedia
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 6 column correspondences:
      name <-> name (score=0.95)
      industry <-> industry (score=0.95)
      country <-> country (score=0.95)
      Sales <-> revenue (score=0.95)
      assets <-> assets (score=0.95)
      id <-> id (score=0.95)
   Entity type: company
   Blocker: token on name
   Renaming columns in Dataset B: {'revenue': 'Sales'}
   Creating PyDI token blocker on column 'name'...
   Materialized 120,057 candidate pairs
    TRUNCATING: 120,057 -> 100,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__industry__industry__exact_match', 0.25), ('sim__country__country__country', 0.15), ('sim__Sales__revenue__numeric_diff', 0.15), ('sim__assets__as

In [2]:
## company datasets dpedia vs fullcontact
fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
dpedia = pd.read_xml('input/datasets/dbpedia.xml')
forbes = pd.read_csv('input/datasets/forbes.csv')


llm = get_llm("gpt")

testset_agent_companies = TrainSetAgent(
    llm=llm,
    output_dir="output/try/score/companies1",
    n_samples=100,              # Total samples to label
    verbose=True,
    target_match_pct=0.2,        # 10% matches (100 matches, 900 non-matches)
    max_candidates=100000,       # Limit candidate pairs to 100k
    min_token_len=4,             # Require tokens of length 4+
    batch_size=10,               # Batch size for labeling
    target_nonmatch_rate=0.6,    # Target 60% non-matches per batch (for hard negatives)
    target_match_rate=0.4,       # Target 40% matches per batch (for hard positives)
    confidence_threshold=0.5     # LLM confidence threshold
)

# Run on fullcontact vs dpedia
gold_standard_ky = testset_agent_companies.run(
    df_a=fullcontact,
    df_b=dpedia,
    pair_name="fullcontact_dpedia",
    id_col_a="id",
    id_col_b="id"
)


Training SET GENERATOR: fullcontact_dpedia
Strategy: BIN-HOPPING (fine bins at top, coarse at bottom)
   Target non-match rate: 60%, Target match rate: 40%
   Confidence threshold: 0.5

   Performing schema matching with LLMBasedSchemaMatcher...
   Found 6 column correspondences:
      id <-> id (score=0.95)
      name <-> name (score=0.95)
      country <-> country (score=0.95)
      city <-> city (score=0.95)
      keypeople <-> keypeople (score=0.95)
      founded <-> founded (score=0.95)
   Entity type: company
   Blocker: token on name
   Creating PyDI token blocker on column 'name'...
   Materialized 136,217 candidate pairs
    TRUNCATING: 136,217 -> 100,000
   Computing similarity scores for 5 column pairs...
   Weighted scores computed: [('sim__name__name__jaro_winkler', 0.35), ('sim__country__country__country', 0.2), ('sim__city__city__token_overlap', 0.15), ('sim__founded__founded__date_diff', 0.15), ('sim__keypeople__keypeople__token_overlap', 0.1)]
      sim__name__name__j